# Delta Lake com PySpark

Notebook de referência para configurar Spark + Delta Lake e executar carga inicial (gravada com `overwrite`), operações `UPDATE` e `DELETE`, além de consultar o histórico de versões e *time travel* no mesmo cenário Statcast do projeto.

## Configuração do Spark com Delta Lake

**O que este bloco faz:** cria o `SparkSession` com `spark.jars.packages` apontando para o artefato Maven do Delta compatível com Spark 3.5 (`io.delta:delta-spark_2.12:3.2.0`), registra as extensões SQL do Delta (`DeltaSparkSessionExtension`), associa o catálogo Spark ao `DeltaCatalog` e fixa o warehouse em `../data/warehouse`. Esse modo evita o bootstrap via `configure_spark_with_delta_pip`, que em WSL sobre `/mnt/c` costuma ficar muito lento ou parecer travado ao resolver/baixar dependências.

In [9]:
import os
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

from pyspark.sql import SparkSession

DELTA_PACKAGES = "io.delta:delta-spark_2.12:3.2.0"

spark = (
    SparkSession.builder.appName("DeltaLake_SATC")
    .config("spark.jars.packages", DELTA_PACKAGES)
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.warehouse.dir", "../data/warehouse")
    .getOrCreate()
)


26/04/29 20:14:11 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Carga inicial da tabela Delta (Statcast)

**O que este bloco faz:** inclui a raiz do projeto no `sys.path`, importa `ler_e_limpar_dados` e `obter_esquema_statcast` de `src/ingestao.py`, lê o CSV bruto em `data/raw/statcast_data.csv`, valida o número de colunas contra o esquema, grava a tabela em formato Delta em `data/delta_statcast` com modo `overwrite` e exibe uma amostra das linhas.

Contexto: estatísticas agregadas da MLB Statcast; `ler_e_limpar_dados` aplica *cast* e renomeia colunas para o modelo **DESEMPENHO_ARREMESSADOR**. Os dados ficam em Parquet com log transacional do Delta, habilitando operações ACID e evolução controlada sem reprocessar o arquivo fonte a cada ajuste pontual.

In [10]:
import sys
from pathlib import Path

raiz_projeto = Path("..").resolve()
if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

from src.ingestao import obter_esquema_statcast, ler_e_limpar_dados

CAMINHO_CSV = str(raiz_projeto / "data" / "raw" / "statcast_data.csv")
CAMINHO_DELTA = str(raiz_projeto / "data" / "delta_statcast")

esquema_statcast = obter_esquema_statcast()
df = ler_e_limpar_dados(spark, CAMINHO_CSV)
assert len(df.columns) == len(esquema_statcast.fields)

df.write.format("delta").mode("overwrite").save(CAMINHO_DELTA)

print("Tabela Delta Statcast gravada em:", CAMINHO_DELTA)
df.show(5, truncate=False)

Tabela Delta Statcast gravada em: /mnt/c/source/trabalho1ed/data/delta_statcast
+----------------+----------------+----------------+---------+----------------------+----------+-----------------+-------------+
|nome_jogador    |total_arremessos|velocidade_media|taxa_giro|media_rebatidas_contra|strikeouts|home_runs_cedidos|walks_cedidos|
+----------------+----------------+----------------+---------+----------------------+----------+-----------------+-------------+
|Webb, Logan     |3282            |88.9            |2038     |0.264                 |224       |14               |46           |
|Rodón, Carlos   |3212            |89.6            |2321     |0.188                 |203       |22               |73           |
|Crochet, Garrett|3151            |92.2            |2443     |0.217                 |255       |24               |46           |
|Gallen, Zac     |3135            |88.8            |2222     |0.239                 |175       |31               |66           |
|Fried, Max      

## UPDATE no Delta Lake (correção de `velocidade_media`)

**O que este bloco faz:** obtém a tabela Delta já gravada com `DeltaTable.forPath`, executa `update` com condição em `nome_jogador` e altera apenas `velocidade_media` via `F.lit`, e por fim filtra e exibe a linha do arremessador para conferência.

Cenário: após revisão de telemetria ou calibração de radar, corrige-se a **velocidade média** sem regravar o CSV inteiro. O predicado localiza o registro e o restante do *pipeline* permanece estável.

In [11]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

tabela_delta = DeltaTable.forPath(spark, CAMINHO_DELTA)

print("Ajustando velocidade_media de Webb, Logan após revisão de medição...")
tabela_delta.update(
    condition="nome_jogador = 'Webb, Logan'",
    set={"velocidade_media": F.lit(89.2)},
)

tabela_delta.toDF().filter(F.col("nome_jogador") == F.lit("Webb, Logan")).select(
    "nome_jogador", "velocidade_media", "total_arremessos", "strikeouts"
).show(truncate=False)

Ajustando velocidade_media de Webb, Logan após revisão de medição...
+------------+----------------+----------------+----------+
|nome_jogador|velocidade_media|total_arremessos|strikeouts|
+------------+----------------+----------------+----------+
|Webb, Logan |89.2            |3282            |224       |
+------------+----------------+----------------+----------+



## INSERT no Delta Lake (novo jogador de teste)

**O que este bloco faz:** insere um registro sintético na tabela Delta para evidenciar operação de **INSERT** via Spark SQL e consulta o mesmo jogador para confirmação imediata da gravação.

**Teoria:** o `INSERT` cria uma nova versão da tabela no log transacional do Delta, mantendo rastreabilidade junto com `UPDATE` e `DELETE`.

In [12]:
spark.sql(
    f"""
    INSERT INTO delta.`{CAMINHO_DELTA}`
    VALUES (
      'Pitcher, Teste',
      500,
      93.5,
      2400,
      0.210,
      110,
      10,
      30
    )
    """
)

spark.sql(
    f"""
    SELECT nome_jogador, velocidade_media, strikeouts
    FROM delta.`{CAMINHO_DELTA}`
    WHERE nome_jogador = 'Pitcher, Teste'
    """
).show(truncate=False)

+--------------+----------------+----------+
|nome_jogador  |velocidade_media|strikeouts|
+--------------+----------------+----------+
|Pitcher, Teste|93.5            |110       |
+--------------+----------------+----------+



## DELETE no Delta Lake (sanção ou linha inválida)

**O que este bloco faz:** reutiliza o objeto `tabela_delta`, chama `delete` com expressão no `nome_jogador` e consulta o DataFrame atual para mostrar como ficaram **Webb, Logan** e **Rodón, Carlos** após a remoção (a linha removida deixa de aparecer).

Cenário: **remoção por sanção**, duplicidade ou registro que não deve permanecer na camada curada. O Delta registra a operação no log para auditoria e *time travel*.

In [13]:
import pyspark.sql.functions as F

print("Removendo estatísticas de Rodón, Carlos (cenário de sanção / expurgo)...")
tabela_delta.delete(condition=F.expr("nome_jogador = 'Rodón, Carlos'"))

tabela_delta.toDF().filter(
    F.col("nome_jogador").isin("Webb, Logan", "Rodón, Carlos")
).select("nome_jogador", "velocidade_media", "strikeouts").show(truncate=False)

Removendo estatísticas de Rodón, Carlos (cenário de sanção / expurgo)...
+------------+----------------+----------+
|nome_jogador|velocidade_media|strikeouts|
+------------+----------------+----------+
|Webb, Logan |89.2            |224       |
+------------+----------------+----------+



## Histórico de versões e time travel (auditoria Delta)

**O que este bloco faz:** projeta o histórico da tabela com `history()` (versão, instante, operação e parâmetros), depois lê a mesma tabela Delta na **versão 0** com `versionAsOf` para recuperar o estado inicial dos jogadores filtrados e confrontar com as mudanças aplicadas pelos `UPDATE` e `DELETE`.

In [14]:
import pyspark.sql.functions as F

print("Histórico de transações (Statcast):")
tabela_delta.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)

print("Time travel: versão 0 — Webb, Logan e Rodón, Carlos como na carga inicial")
df_versao_zero = (
    spark.read.format("delta")
    .option("versionAsOf", 0)
    .load(CAMINHO_DELTA)
    .filter(F.col("nome_jogador").isin("Webb, Logan", "Rodón, Carlos"))
    .select("nome_jogador", "velocidade_media", "strikeouts")
    .orderBy("nome_jogador")
)
df_versao_zero.show(truncate=False)

Histórico de transações (Statcast):
+-------+-----------------------+---------+------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                   |
+-------+-----------------------+---------+------------------------------------------------------+
|7      |2026-04-29 20:14:20.848|DELETE   |{predicate -> ["(nome_jogador#5709 = Rodón, Carlos)"]}|
|6      |2026-04-29 20:14:17.548|WRITE    |{mode -> Append, partitionBy -> []}                   |
|5      |2026-04-29 20:14:15.227|UPDATE   |{predicate -> ["(nome_jogador#5709 = Webb, Logan)"]}  |
|4      |2026-04-29 20:14:12.815|WRITE    |{mode -> Overwrite, partitionBy -> []}                |
|3      |2026-04-29 19:46:18.659|DELETE   |{predicate -> ["(nome_jogador#594 = Rodón, Carlos)"]} |
|2      |2026-04-29 19:46:11.955|WRITE    |{mode -> Append, partitionBy -> []}                   |
|1      |2026-04-29 19:46:04.314|UPDATE   |{predicate -> ["(nome_jogador#